In [ ]:
%%time
# Imports principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from skimage.io import imread
from skimage.transform import resize
import warnings
warnings.filterwarnings('ignore') # Limpiar advertencias

# Preprocesamiento y Métricas
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.preprocessing import StandardScaler, label_binarize, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc, classification_report
from sklearn.pipeline import Pipeline

# --- OPTIMIZACIÓN CON OPTUNA (BO-TPE) ---
import optuna

# --- CLASIFICADORES ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

In [ ]:
# Función carga
def load_image_dataset(root_dir, size=(64,64), max_per_class=None):
    X, y = [], []
    classes = sorted(os.listdir(root_dir))
    for cls in classes:
        cls_path = os.path.join(root_dir, cls)
        if not os.path.isdir(cls_path): continue
        files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg'))]
        if max_per_class: files = files[:max_per_class]
        for f in files:
            img = imread(os.path.join(cls_path, f))
            if img.ndim == 3: img = img[...,0]
            img_resized = resize(img, size, anti_aliasing=True)
            X.append(img_resized)
            y.append(cls)
    return np.array(X), np.array(y), classes

# Carga y Proceso
X, y_str, classes = load_image_dataset("CMS_data", size=(64,64))
print("Datos cargados.")

X = X / X.max()
X_flat = X.reshape(len(X), -1)

# Label Encoding (Strings -> Enteros)
le = LabelEncoder()
y_int = le.fit_transform(y_str)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y_int, test_size=0.2, stratify=y_int, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Binarizar para ROC
y_test_bin = label_binarize(y_test, classes=np.arange(len(classes)))

In [ ]:
def plot_multiclass_roc(y_test_bin, y_score, classes, title):
    plt.figure(figsize=(8,6))
    for i in range(len(classes)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{classes[i]} (AUC = {roc_auc:.2f})")
    plt.plot([0,1], [0,1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

def plot_learning_curve(estimator, title, X, y, ylim=None, cv=3, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)):
    plt.figure(figsize=(10, 6))
    plt.title(title)
    if ylim is not None: plt.ylim(*ylim)
    plt.xlabel("Ejemplos de entrenamiento")
    plt.ylabel("Score (Accuracy)")
    
    train_sizes_abs, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes, scoring="accuracy"
    )
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    
    plt.grid(True)
    plt.fill_between(train_sizes_abs, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes_abs, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes_abs, train_scores_mean, 'o-', color="r", label="Train Score")
    plt.plot(train_sizes_abs, test_scores_mean, 'o-', color="g", label="CV Score")
    plt.legend(loc="best")
    plt.show()

In [ ]:
%%time
print("Optimizando KNN con Optuna (TPE)...")

def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 3, 20)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    
    clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, n_jobs=-1)
    return cross_val_score(clf, X_train, y_train, n_jobs=-1, cv=3).mean()

study_knn = optuna.create_study(direction='maximize')
study_knn.optimize(objective_knn, n_trials=15)

print(f"Mejores parámetros KNN: {study_knn.best_params}")

In [ ]:
best_knn = KNeighborsClassifier(n_jobs=-1, **study_knn.best_params)
best_knn.fit(X_train, y_train)

y_pred_knn = best_knn.predict(X_test)
y_score_knn = best_knn.predict_proba(X_test)

print("--- Reporte KNN (Optuna) ---")
print(classification_report(y_test, y_pred_knn, target_names=classes))

# Matriz
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_knn), display_labels=classes)
disp.plot(cmap="Oranges", xticks_rotation=45)
plt.title("Matriz KNN (Optuna)")
plt.show()

# Precisión
pd.DataFrame(classification_report(y_test, y_pred_knn, output_dict=True, target_names=classes)).transpose().loc[classes, "precision"].plot(kind="bar", color="orange")
plt.title("Precisión KNN")
plt.show()

# ROC
plot_multiclass_roc(y_test_bin, y_score_knn, classes, "ROC - KNN (Optuna)")

# Learning Curve
plot_learning_curve(best_knn, "Curva de Aprendizaje - KNN", X_train, y_train, cv=3, ylim=(0.4, 1.0))

In [ ]:
best_knn = KNeighborsClassifier(n_jobs=-1, **study_knn.best_params)
best_knn.fit(X_train, y_train)

y_pred_knn = best_knn.predict(X_test)
y_score_knn = best_knn.predict_proba(X_test)

print("--- Reporte KNN (Optuna) ---")
print(classification_report(y_test, y_pred_knn, target_names=classes))

In [ ]:
# Matriz
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_knn), display_labels=classes)
disp.plot(cmap="Oranges", xticks_rotation=45)
plt.title("Matriz KNN (Optuna)")
plt.show()

In [ ]:
# Precisión
pd.DataFrame(classification_report(y_test, y_pred_knn, output_dict=True, target_names=classes)).transpose().loc[classes, "precision"].plot(kind="bar", color="orange")
plt.title("Precisión KNN")
plt.show()

In [ ]:
# ROC
plot_multiclass_roc(y_test_bin, y_score_knn, classes, "ROC - KNN (Optuna)")

In [ ]:
# Learning Curve
plot_learning_curve(best_knn, "Curva de Aprendizaje - KNN", X_train, y_train, cv=3, ylim=(0.4, 1.0))